v

In [1]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

/workspaces/ai-agents-for-beginners/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [3]:
@tool(approval_mode="never_require")
def check_if_read(
    emails: Annotated[str, "The email to check if it is read"]
) -> str:
    """Check if an email has been read by the user before."""
    email_list = {
        "Email1": True,
        "Email2": True,
        "Email3": True,
        "Email4": True,
        "Email5": False,
    }
    is_read = email_list.get(emails, False)
    return f"{emails} is {'read' if is_read else 'not read'}."

@tool(approval_mode="never_require")
def get_inbox_emails() -> list[str]:
    """Retrieves the list of all email IDs in the user's inbox."""
    return ["Email1", "Email2", "Email3", "Email4", "Email5"]

In [4]:
agent = await provider.create_agent(
    name="EmailSortingAgent",
    instructions=(
        "You are an email sorting agent. Help users check if their emails are read or not "
        "and try to motivate the user to read their un-read emails."
    ),
    tools=[check_if_read, get_inbox_emails],
)

In [5]:
session = agent.create_session()


response = await agent.run(
    "Do i have any emails that i have not looked at?",
    session=session,
)
print(f"Agent: {response}")

Agent: You have 5 emails in your inbox. Out of these, 4 have already been read, but there is one unread email: **Email5**.

It might contain something important or interesting, so I recommend you check it out soon! 😊


In [10]:
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

# Ζητάμε άδεια ΜΟΝΟ για ανάγνωση (Read-only). Είναι 100% ασφαλές.
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def get_real_unread_emails():
    """Συνδέεται στο Gmail και τυπώνει τα τελευταία αδιάβαστα emails."""
    creds = None
    
    # Το token.json αποθηκεύει την άδειά μας για να μην κάνουμε login κάθε φορά
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
        
    # Αν δεν έχουμε άδεια (πρώτη φορά), κάνουμε login!
    # Αν δεν έχουμε άδεια (πρώτη φορά), κάνουμε login!
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            flow.redirect_uri = 'http://localhost'
            auth_url, _ = flow.authorization_url(prompt='consent')
            
            print("\n" + "="*60)
            print("1️⃣ ΠΑΤΑ ΑΥΤΟ ΤΟ LINK ΓΙΑ ΝΑ ΚΑΝΕΙΣ LOGIN ΣΤΗ GOOGLE:")
            print(auth_url)
            print("="*60 + "\n")
            
            print("2️⃣ Θα σε πάει στη σελίδα Login. Αφού πατήσεις 'Συνεχίστε', θα βγάλει πάλι το σφάλμα 'This site can't be reached'.")
            print("3️⃣ ΕΙΝΑΙ ΦΥΣΙΟΛΟΓΙΚΟ! Κάνε ΑΝΤΙΓΡΑΦΗ όλο το link από την πάνω μπάρα (που ξεκινάει με http://localhost...)")
            
            # Εδώ ο κώδικας σταματάει και περιμένει να του δώσεις το link!
            response_url = input("\n4️⃣ Κάνε ΕΠΙΚΟΛΛΗΣΗ το link εδώ και πάτα Enter: ")
            
            # Διαβάζει το link που του έδωσες και παίρνει το κλειδί
            flow.fetch_token(authorization_response=response_url.strip())
            creds = flow.credentials
            
        # Αποθηκεύουμε την άδεια για την επόμενη φορά
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    # Συνδεόμαστε στο Gmail API!
    service = build('gmail', 'v1', credentials=creds)

    print("\nΣυνδέθηκε με επιτυχία! Ψάχνω για αδιάβαστα emails...\n")
    
    # Ζητάμε τα τελευταία 3 αδιάβαστα emails
    results = service.users().messages().list(userId='me', q='is:unread', maxResults=3).execute()
    messages = results.get('messages', [])

    if not messages:
        print("Δεν βρήκα κανένα αδιάβαστο email. Το Inbox σου είναι καθαρό!")
        return

    print("=== Τα Τελευταία σου Αδιάβαστα Emails ===")
    for msg in messages:
        # Παίρνουμε τις λεπτομέρειες κάθε email
        msg_detail = service.users().messages().get(userId='me', id=msg['id'], format='metadata', metadataHeaders=['Subject', 'From']).execute()
        headers = msg_detail.get('payload', {}).get('headers', [])
        
        subject = next((header['value'] for header in headers if header['name'] == 'Subject'), 'Χωρίς Θέμα')
        sender = next((header['value'] for header in headers if header['name'] == 'From'), 'Άγνωστος Αποστολέας')
        
        print(f"Από: {sender}")
        print(f"Θέμα: {subject}")
        print("-" * 40)

# Ώρα να το τρέξουμε!
get_real_unread_emails()


1️⃣ ΠΑΤΑ ΑΥΤΟ ΤΟ LINK ΓΙΑ ΝΑ ΚΑΝΕΙΣ LOGIN ΣΤΗ GOOGLE:
https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1014914619663-c9t2m05rjt82aisr1fp7v1iclao6uce1.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=vhIBokaLGOnfdctcJv3xoPZnlvYqts&code_challenge=mZLLdpKS75cbEmAsKrdMPOBjCfhqf8wbwbnjyswgCL8&code_challenge_method=S256&prompt=consent&access_type=offline

2️⃣ Θα σε πάει στη σελίδα Login. Αφού πατήσεις 'Συνεχίστε', θα βγάλει πάλι το σφάλμα 'This site can't be reached'.
3️⃣ ΕΙΝΑΙ ΦΥΣΙΟΛΟΓΙΚΟ! Κάνε ΑΝΤΙΓΡΑΦΗ όλο το link από την πάνω μπάρα (που ξεκινάει με http://localhost...)

Συνδέθηκε με επιτυχία! Ψάχνω για αδιάβαστα emails...

=== Τα Τελευταία σου Αδιάβαστα Emails ===
Από: Google <no-reply@google.com>
Θέμα: Information about your new Google Account
----------------------------------------
